<a href="https://colab.research.google.com/github/Shineii86/PairExtraordinaire/blob/main/notebooks/PairExtraordinaire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
  <img src="https://raw.githubusercontent.com/Shineii86/PairExtraordinaire/refs/heads/main/images/PairExtraordinaire.png" width="300px" alt="Pair Extraordinaire Achievement">
  <h1>👥 Pair Extraordinaire Automator</h1>
  <p><b>Automate the creation of co-authored commits and merged pull requests to unlock all tiers of the Pair Extraordinaire achievement — all from your browser.</b></p>
</div>

---

> ⚠️ **WARNING**
> - This script will create real co-authored commits and merge pull requests on your GitHub account.
> - **Never share your Personal Access Token** – treat it like a password.
> - Use a **dedicated repository** to avoid cluttering important projects.
> - Your co-author must have **write access** to the repository. You can add them as a collaborator in the repo settings.

---

In [ ]:
#@title 📦 1. Install Dependencies & Setup
!pip install -q PyGithub

import time
import random
import string
from github import Github, GithubException

print("✅ All dependencies installed and imported.")

In [ ]:
#@title ⚙️ 2. Configuration & Run

# =============================================================================
#  🔧 Configuration
# =============================================================================

# Your GitHub username
GITHUB_USERNAME = "Shineii86"  #@param {type:"string"}

# Your Personal Access Token (keep secret!)
GITHUB_TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"  #@param {type:"string"}

# Repository name (must exist under your account)
REPO_NAME = "PairExtraordinaire"  #@param {type:"string"}

# Base branch to target (default: main)
BASE_BRANCH = "main"  #@param {type:"string"}

# Co-author's GitHub username (e.g., "collaborator")
COAUTHOR_USERNAME = "collaborator"  #@param {type:"string"}

# Co-author's email (must be the email associated with their GitHub account)
COAUTHOR_EMAIL = "collaborator@example.com"  #@param {type:"string"}

# Co-author's full name (optional, used in the commit message)
COAUTHOR_NAME = "Collaborator Name"  #@param {type:"string"}

# Number of pull requests to create and merge (minimum 1 for base achievement)
# Tier requirements: Base=1, Bronze=10, Silver=24, Gold=48
NUM_PRS = 1  #@param {type:"integer"}

# Delay in seconds between PRs (default: 10)
DELAY_SECONDS = 10  #@param {type:"integer"}

# Maximum retries for merge failures (default: 3)
MAX_RETRIES = 3  #@param {type:"integer"}

# =============================================================================
#  🚀 Automation Script (Runs automatically after configuration)
# =============================================================================

print(f"\nConfiguration loaded for user '{GITHUB_USERNAME}' on repo '{REPO_NAME}'")
print(f"Base branch: {BASE_BRANCH}")
print(f"Co-author: {COAUTHOR_USERNAME} <{COAUTHOR_EMAIL}>")
print(f"Will create {NUM_PRS} co-authored pull request(s) with {DELAY_SECONDS}s delay.\n")

def generate_random_string(length=8):
    return ''.join(random.choices(string.ascii_lowercase, k=length))

def wait_for_mergeability(pr, max_wait=30):
    """Wait until GitHub confirms the PR is mergeable (or timeout)."""
    waited = 0
    while waited < max_wait:
        pr.update()
        if pr.mergeable is not False:
            return True
        time.sleep(3)
        waited += 3
    return False

# Connect to GitHub
g = Github(GITHUB_TOKEN)
repo = g.get_user().get_repo(REPO_NAME)

successful_merges = 0

for i in range(NUM_PRS):
    print(f"\n--- 📦 Creating PR #{i+1} of {NUM_PRS} ---")

    # 1. Get latest commit SHA from base branch
    base_branch_obj = repo.get_branch(BASE_BRANCH)
    latest_commit_sha = base_branch_obj.commit.sha
    print(f"📌 Latest {BASE_BRANCH} commit: {latest_commit_sha[:7]}")

    # 2. Create a new branch
    branch_name = f"pair-extra-{generate_random_string(6)}-{int(time.time())}"
    repo.create_git_ref(f"refs/heads/{branch_name}", latest_commit_sha)
    print(f"✅ Created branch: {branch_name}")

    # 3. Prepare co-authored commit
    co_author_line = f"Co-authored-by: {COAUTHOR_NAME} <{COAUTHOR_EMAIL}>"
    commit_message = f"feat: pair extraordinaire #{i+1}\n\n{co_author_line}"

    file_content = f"# {REPO_NAME}\n\nThis commit was co-authored by @{GITHUB_USERNAME} and @{COAUTHOR_USERNAME}.\n\n- PR #{i+1} of {NUM_PRS}\n- Auto-generated: {generate_random_string()}\n\n---\n🤖 Made By [Shinei Nouzen](https://github.com/Shineii86/PairExtraordinaire)"

    # 4. Create or update a file with the co-authored commit
    try:
        contents = repo.get_contents("README.md", ref=branch_name)
        repo.update_file(
            contents.path,
            commit_message,
            f"{contents.decoded_content.decode()}\n\n{file_content}",
            contents.sha,
            branch=branch_name
        )
        print("📝 Updated README.md with co-authored commit")
    except GithubException:
        repo.create_file(
            "README.md",
            commit_message,
            file_content,
            branch=branch_name
        )
        print("📄 Created README.md with co-authored commit")

    print(f"👥 {co_author_line}")

    # 5. Create Pull Request
    pr = repo.create_pull(
        title=f"Pair Extraordinaire #{i+1}: Co-authored by @{GITHUB_USERNAME} and @{COAUTHOR_USERNAME}",
        body=f"🤖 Automated pull request #{i+1} of {NUM_PRS} for the Pair Extraordinaire achievement.\n\nThis PR contains a co-authored commit.\n\n---\n👥 PR Automation By [Shinei Nouzen](https://github.com/Shineii86)",
        head=branch_name,
        base=BASE_BRANCH
    )
    print(f"🔗 Created PR: {pr.html_url}")

    # 6. Wait for mergeability
    print("⏳ Waiting for GitHub to calculate mergeability...")
    if not wait_for_mergeability(pr):
        print("❌ PR not mergeable after waiting. Stopping.")
        break

    # 7. Merge PR with retry logic
    merge_success = False
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            pr.merge(merge_method="merge")
            print(f"🎉 Merged PR #{pr.number} on attempt {attempt}")
            successful_merges += 1
            merge_success = True
            break
        except GithubException as e:
            print(f"❌ Merge attempt {attempt}/{MAX_RETRIES} failed: {e}")
            if attempt < MAX_RETRIES:
                print(f"⏳ Waiting {DELAY_SECONDS}s before retry...")
                time.sleep(DELAY_SECONDS)
                pr.update()

    if not merge_success:
        print("❌ All merge attempts exhausted. Stopping.")
        break

    # 8. Delay before next PR
    if i < NUM_PRS - 1:
        print(f"⏸️ Pausing {DELAY_SECONDS} seconds for GitHub to process the merge...")
        time.sleep(DELAY_SECONDS)

print(f"\n🏁 Finished. Successfully merged {successful_merges} out of {NUM_PRS} co-authored pull requests.")

tier_info = {
    1: "Base",
    10: "Bronze",
    24: "Silver",
    48: "Gold"
}
achieved_tier = None
for threshold, tier in sorted(tier_info.items()):
    if successful_merges >= threshold:
        achieved_tier = tier

if achieved_tier:
    print(f"👥 Congratulations! You've met the requirements for Pair Extraordinaire ({achieved_tier})!")
else:
    print("⚠️ You need at least 1 merged co-authored PR for the base achievement.")

---

### 📚 Need Help?

- **Achievement not appearing?** Wait a few minutes and refresh your GitHub profile.
- **"405 Not mergeable" error?** Enable **"Allow auto-merge"** in repo **Settings → General → Pull Requests**.
- **Bad credentials?** Double‑check your Personal Access Token and ensure it has `repo` scope.

---

<div align="center">

**Copyright [Shinei Nouzen](https://github.com/Shineii86) All Rights Reserved.**  
*Made with ❤️ for GitHub achievement hunters*


***Found this useful? Give it a Star! ⭐ [Shineii86/PairExtraordinaire](https://github.com/Shineii86/PairExtraordinaire)***

</div>